# V2 Colab training and validation

Mount Google Drive, install the pinned v2 dependencies, run the fixture check, and train and validate LightGBM, XGBoost, and CatBoost in sequence. The three estimators share the expensive data import, candidate, split, and feature stages; each keeps its own model checkpoints, config, metrics, and report. The comparison uses the challenge metric, per-entity macro F₀.₅. It does not use test labels or outside business data.

The notebook assumes the repository is in `MyDrive/amazon-ml` and the supplied train/test files follow the repository's `student_resource/dataset/{train,test}` layout. Edit the path settings below if yours differs. Training outputs and resumable work are saved under the repository's `code/business_entity_resolution/artifacts/v2_colab/`.

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

# Change this if you placed the repository elsewhere in MyDrive.
PROJECT_DIR = Path('/content/drive/MyDrive/amazon-ml')
PACKAGE = PROJECT_DIR / 'code/business_entity_resolution'
SRC = PACKAGE / 'src'
TRAIN_DIR = PROJECT_DIR / 'student_resource/dataset/train'
TEST_DIR = PROJECT_DIR / 'student_resource/dataset/test'
ARTIFACTS = PACKAGE / 'artifacts/v2_colab'
MODELS = ('lightgbm', 'xgboost', 'catboost')
RUN_TEST_INFERENCE = False  # optional; test labels are never read

required_files = [
    *(TRAIN_DIR / name for name in ('train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv')),
    *(TEST_DIR / name for name in ('test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv')),
]
missing = [str(path) for path in required_files if not path.is_file()]
if not (SRC / 'train_v2.py').is_file():
    raise FileNotFoundError(f'V2 source code not found under {PROJECT_DIR}')
if missing:
    raise FileNotFoundError('Missing challenge files:\n' + '\n'.join(missing))

ARTIFACTS.mkdir(parents=True, exist_ok=True)
print(f'Project: {PROJECT_DIR}')
print(f'Training data: {TRAIN_DIR}')
print(f'Test data: {TEST_DIR}')
print(f'Artifacts: {ARTIFACTS}')
print(f'Colab-visible CPU cores: {os.cpu_count() or 1}')
print(f'Free space at artifact location: {shutil.disk_usage(ARTIFACTS).free / 2**30:.1f} GiB')

## Install pinned requirements

This installs the base pipeline and the optional estimator packages. The scripts run as child processes, so the packages are imported after installation.

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(PACKAGE / 'requirements.txt'),
    '-r', str(PACKAGE / 'requirements-models.txt'),
])
importlib.invalidate_caches()
for module in ('duckdb', 'numpy', 'lightgbm', 'xgboost', 'catboost', 'rapidfuzz'):
    if importlib.util.find_spec(module) is None:
        raise ModuleNotFoundError(f'{module} is still unavailable after dependency installation')
print('Pinned dependencies are installed for all three models.')

## Fixture check and three-model validation

The full-corpus run is lengthy. Run the notebook from this cell through the report cell. The shared work directory on Drive lets later estimators reuse completed data stages. Estimator checkpoints and final artifacts stay isolated by model.

In [ ]:
def run_step(label, script, *args):
    command = [sys.executable, str(script), *(str(arg) for arg in args)]
    print(f"\n=== {label} ===\n$ " + ' '.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, command)
    print(f'=== {label} finished ===', flush=True)

run_step('V2 fixture check', SRC / 'test_v2.py')

for model in MODELS:
    run_step(f'{model}: full-corpus training and held-out validation',
             SRC / 'train_v2.py', '--train-dir', TRAIN_DIR,
             '--artifacts-dir', ARTIFACTS, '--model', model)
    run_step(f'{model}: render validation report',
             SRC / 'report_v2.py', '--artifacts-dir', ARTIFACTS, '--model', model)

## Compare and verify validation artifacts

Each report and metric file is written to its estimator's own directory. This cell also writes a compact CSV and Markdown comparison.

In [ ]:
import csv
import json

def model_dir(name):
    return ARTIFACTS if name == 'lightgbm' else ARTIFACTS / 'models' / name

comparison = []
for name in MODELS:
    folder = model_dir(name)
    metrics_path = folder / 'training_metrics.json'
    report_path = folder / 'validation_report.md'
    config_path = folder / 'model_config.json'
    model_path = folder / 'matcher_model.joblib'
    if not all(path.is_file() for path in (metrics_path, report_path, config_path, model_path)):
        raise FileNotFoundError(f'{name} is missing one or more expected outputs in {folder}')
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if config.get('estimator') != name:
        raise ValueError(f'{folder} contains estimator {config.get("estimator")}, expected {name}')
    validation = metrics['final_validation']
    comparison.append({
        'model': name,
        'validation_macro_f05': validation['macro_f05'],
        'pair_precision': validation['pair_precision'],
        'pair_recall': validation['pair_recall'],
        'candidate_cap_per_source': config['candidate_cap_per_source'],
        'decision_threshold': config['decision_threshold'],
        'feature_count': len(config['feature_names']),
        'artifact_directory': str(folder),
    })

fieldnames = list(comparison[0])
with (ARTIFACTS / 'model_comparison.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(comparison)
lines = ['# V2 model validation comparison', '',
         'Primary metric: held-out per-entity macro F₀.₅, including singleton entities.', '',
         '| Model | Macro F₀.₅ | Pair precision | Pair recall | Candidate cap/source | Threshold | Features | Artifacts |',
         '|---|---:|---:|---:|---:|---:|---:|---|']
for row in comparison:
    lines.append(f"| {row['model']} | {row['validation_macro_f05']:.6f} | {row['pair_precision']:.6f} | {row['pair_recall']:.6f} | {row['candidate_cap_per_source']} | {row['decision_threshold']:.2f} | {row['feature_count']} | `{row['artifact_directory']}` |")
(ARTIFACTS / 'model_comparison.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('\n'.join(lines))
print(f"\nCSV: {ARTIFACTS / 'model_comparison.csv'}")

## Optional: generate separate test outputs

Leave `RUN_TEST_INFERENCE = False` to stop after validation. Set it to `True` in the setup cell and run this cell to generate each model's separate test output folder. This reads test source records only; no test labels are available or used.

In [ ]:
if RUN_TEST_INFERENCE:
    for model in MODELS:
        run_step(f'{model}: test inference and format validation',
                 SRC / 'infer_v2.py', '--test-dir', TEST_DIR,
                 '--artifacts-dir', ARTIFACTS, '--model', model)
else:
    print('Test inference skipped. Set RUN_TEST_INFERENCE = True to run it for all models.')